# Car Detection using a Pre-trained Model

This notebook demonstrates how to detect cars in images using a pre-trained object detection model from TensorFlow Hub. We will use the SSD MobileNet V2 model, which is efficient and provides good accuracy for common objects, including cars.

## 1. Install and Import Dependencies

In [ ]:
# Install necessary libraries (if running in a new environment)
!pip install -q tensorflow tensorflow_hub opencv-python-headless matplotlib

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time

# Optional: For drawing on images
from six import BytesIO
from six.moves.urllib.request import urlopen

## 2. Load the Pre-trained Model

We will load an object detection model from TensorFlow Hub. The SSD MobileNet V2 model is a good choice for a balance of speed and accuracy.

You can find more models on [TensorFlow Hub](https://tfhub.dev/s?module-type=image-object-detection).

In [ ]:
MODEL_HANDLE = 'https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2'
print(f"Loading model from {MODEL_HANDLE}...")
detector = hub.load(MODEL_HANDLE)
print("Model loaded successfully!")

# The model expects a batch of 3-channel images of type tf.uint8 and shape [N, H, W, 3]
# It returns a dictionary with detection boxes, classes, and scores.

## 3. Helper Functions

We'll define a few helper functions to:
- Load an image from a file path.
- Preprocess the image for the model.
- Run inference (detection).
- Draw bounding boxes on the detected objects.

In [ ]:
def load_image_into_numpy_array(path):
    """Load an image from file into a numpy array.

    Puts image into numpy array to feed into tensorflow graph.
    Note that by convention we put it into a numpy array with shape
    (height, width, channels), where channels=3 for RGB.

    Args:
      path: the file path to the image

    Returns:
      uint8 numpy array with shape (img_height, img_width, 3)
    """
    image = None
    if(path.startswith('http')):
        response = urlopen(path)
        image_data = response.read()
        image_data = BytesIO(image_data)
        image = Image.open(image_data)
    else:
        image_data = tf.io.gfile.GFile(path, 'rb').read()
        image = Image.open(BytesIO(image_data))

    (im_width, im_height) = image.size
    return np.array(image.getdata()).reshape(
        (im_height, im_width, 3)).astype(np.uint8)

In [ ]:
def run_detector(detector_model, image_np):
    """Runs inference on a single image.
    
    Args:
      detector_model: A loaded detection model from TensorFlow Hub.
      image_np: A numpy array representing the image (H, W, C).
      
    Returns:
      A dictionary containing detection results.
    """
    # The model expects a batch of images, so add a batch dimension.
    image_tensor = tf.convert_to_tensor(image_np)[tf.newaxis, ...]
    
    start_time = time.time()
    result = detector_model(image_tensor)
    end_time = time.time()
    
    print(f"Inference time: {end_time - start_time:.2f}s")
    
    # The result is a dictionary, convert its values to numpy arrays
    result = {key:value.numpy() for key,value in result.items()}
    return result

In [ ]:
# COCO 2017 dataset category names and IDs (relevant for many TF Hub models)
# We are primarily interested in 'car' (ID 3), 'truck' (ID 8), 'bus' (ID 6)
COCO17_HUMAN_READABLE_NAMES = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

def draw_bounding_boxes(image_np, results, min_score_thresh=0.5, target_class_ids=[3, 6, 8]):
    """Draws bounding boxes on an image.
    
    Args:
      image_np: A numpy array representing the image.
      results: A dictionary of detection results from the model.
      min_score_thresh: Minimum score threshold for displaying a box.
      target_class_ids: List of class IDs to draw boxes for (e.g., [3] for 'car').
                          The SSD MobileNet V2 model uses COCO class IDs.
                          'car': 3, 'motorcycle': 4, 'bus': 6, 'truck': 8
    Returns:
      Image with bounding boxes drawn.
    """
    image_with_boxes = image_np.copy()
    boxes = results['detection_boxes'][0]       # First (and only) image in batch
    classes = results['detection_classes'][0].astype(int)
    scores = results['detection_scores'][0]
    
    im_height, im_width, _ = image_np.shape
    
    num_detections = 0
    for i in range(boxes.shape[0]):
        if scores[i] >= min_score_thresh and classes[i] in target_class_ids:
            num_detections +=1
            ymin, xmin, ymax, xmax = tuple(boxes[i])
            (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                          ymin * im_height, ymax * im_height)
            
            # Draw rectangle
            cv2.rectangle(image_with_boxes, (int(left), int(top)), (int(right), int(bottom)), (0, 255, 0), 2)
            
            # Draw label
            class_name = COCO17_HUMAN_READABLE_NAMES[classes[i]-1] # Class IDs are 1-indexed in COCO
            label = f"{class_name}: {scores[i]:.2f}"
            cv2.putText(image_with_boxes, label, (int(left), int(top) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
    print(f"Detected {num_detections} relevant objects.")
    return image_with_boxes

## 4. Load Test Image and Run Detection

Now, let's load a test image and see the detector in action. 
**Note:** You'll need to provide paths to your own images. We'll add some example images later.

In [ ]:
# Placeholder for image paths - we will add actual images in a later step.
TEST_IMAGE_PATHS = [
    # "path/to/your/test_image_1.jpg", 
    # "path/to/your/test_image_2.jpg"
]

if not TEST_IMAGE_PATHS:
    print("Please add image paths to TEST_IMAGE_PATHS to run the detection.")
    print("Using a default placeholder image from the web for demonstration purposes.")
    # A freely usable image of cars from Wikipedia Commons for placeholder
    TEST_IMAGE_PATHS.append("https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Cars_in_traffic_in_Auckland%2C_New_Zealand_-_copyright-free_photo_released_to_public_domain.jpg/1280px-Cars_in_traffic_in_Auckland%2C_New_Zealand_-_copyright-free_photo_released_to_public_domain.jpg")


for image_path in TEST_IMAGE_PATHS:
    print(f"\nProcessing image: {image_path}")
    try:
        image_np = load_image_into_numpy_array(image_path)
        
        # Display original image
        plt.figure(figsize=(12, 8))
        plt.subplot(1, 2, 1)
        plt.title("Original Image")
        plt.imshow(image_np)
        plt.axis("off")

        # Run detection
        results = run_detector(detector, image_np)
        
        # Draw bounding boxes for cars, trucks, buses
        image_with_boxes = draw_bounding_boxes(image_np, results, min_score_thresh=0.4, target_class_ids=[3, 6, 8]) 
        
        # Display image with detections
        plt.subplot(1, 2, 2)
        plt.title("Detections")
        plt.imshow(image_with_boxes)
        plt.axis("off")
        plt.show()
        
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")

## 5. Saving the Model (Information)

The model we are using (`ssd_mobilenet_v2`) is loaded directly from TensorFlow Hub using its URL handle. This means the model itself isn't downloaded as a single file into our local directory by default when we call `hub.load()`.

**How to "save" or reuse this model:**
1.  **Use the Hub Handle**: The easiest way is to save the `MODEL_HANDLE` string ('https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2') and use `hub.load(MODEL_HANDLE)` whenever you need the model. TensorFlow Hub caches the model locally after the first download, so subsequent loads are fast.
2.  **Save as SavedModel format**: If you need to save the model to a specific path (e.g., for deployment without internet access or for conversion to other formats), you can do so using `tf.saved_model.save()`.

   ```python
   # Example of saving the loaded model
   # model_save_path = "Car Detection/saved_model/ssd_mobilenet_v2"
   # tf.saved_model.save(detector, model_save_path)
   # print(f"Model saved to {model_save_path}")
   
   # To load it back:
   # reloaded_detector = tf.saved_model.load(model_save_path)
   ```
For this notebook, we primarily rely on loading from TF Hub. If you uncomment and run the cell above, the model will be saved into the `Car Detection/saved_model/ssd_mobilenet_v2` directory.

In [ ]:
# Code to demonstrate saving the model (optional)
SAVE_MODEL_LOCALLY = True # Set to True to save the model files

if SAVE_MODEL_LOCALLY:
    # Ensure the save path is within the 'Car Detection' directory
    model_save_path = "Car Detection/saved_model/ssd_mobilenet_v2_detector"
    print(f"Saving model to ./{model_save_path}...")
    # Ensure the detector is a trackable object if it's a function-like Keras model
    # For TF Hub models loaded with hub.load, they are often already trackable.
    # If it's a Keras Layer, it's directly savable.
    # If it's a signature, you might need to wrap it or save specific signatures.
    
    # The 'detector' object from hub.load is directly usable with tf.saved_model.save
    tf.saved_model.save(detector, model_save_path)
    print(f"Model saved to ./{model_save_path}")
    
    # You can then load it back using:
    # reloaded_model = tf.saved_model.load(model_save_path)
    # And use it like: 
    # detections = reloaded_model(image_tensor)
else:
    print("Skipping local model saving. The model is loaded from TensorFlow Hub.")

## Next Steps

- Try with your own images!
- Experiment with different models from TensorFlow Hub.
- Adjust the `min_score_thresh` to see more or fewer detections.
- Modify `target_class_ids` to detect other objects.